In [1]:
# =====================================
#  Preprocessing – SINGLE LANGUAGE ONLY
# =====================================

import json
import pandas as pd
from tqdm import tqdm
import re
import os
import random
from sklearn.model_selection import train_test_split

# Configuration
DATASET_PATH = "/kaggle/input/llm-training-in-multilingual-languages"
MAX_PROMPT_TOKENS = 200
MAX_TARGET_TOKENS = 300
MIN_RESPONSE_LENGTH = 20

def load_jsonl_safe(filepath):
    records = []
    bad_lines = 0
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1
    print(f"{filepath} → loaded {len(records)} rows, skipped {bad_lines} bad lines")
    return pd.DataFrame(records)

def clean_text(text):
    text = text.replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

def estimate_tokens(text):
    return len(text.split())

def check_language_script(text, language):
    if language == "en":
        return bool(re.search(r'[a-zA-Z]', text))
    elif language == "si":
        return bool(re.search(r'[\u0D80-\u0DFF]', text))
    elif language == "ta":
        return bool(re.search(r'[\u0B80-\u0BFF]', text))
    return True

In [2]:
# === ENGLISH ONLY ===
LANGUAGE = "en"
FILE_NAME = "english_dataset.jsonl"

# Load
print(f"Loading {LANGUAGE} dataset...")
df = load_jsonl_safe(os.path.join(DATASET_PATH, FILE_NAME))
df["language"] = LANGUAGE

# Add rest of preprocessing (cleaning, filtering, etc.)
print("\nCleaning text...")
df["user_input"] = df["user_input"].apply(clean_text)
df["bot_response"] = df["bot_response"].apply(clean_text)
df["user_emotion"] = df["user_emotion"].str.lower().str.strip()

# Length filters
df = df[df["user_input"].str.len() > 5]
df = df[df["bot_response"].str.len() > MIN_RESPONSE_LENGTH]

# Remove duplicates
df = df.drop_duplicates(subset=["user_input", "bot_response"])

# Script validation
df["valid_script"] = df.apply(
    lambda row: check_language_script(row["user_input"], LANGUAGE) and
                check_language_script(row["bot_response"], LANGUAGE), axis=1)
df = df[df["valid_script"]].drop(columns=["valid_script"])

print(f"After filtering: {len(df)} rows")

def build_prompt(row):
    # We use the emotion ONLY to guide the model, not to be part of its output.
    # We include it in the system instruction so the model "knows" the context.
    emotion = row["user_emotion"]
    
    # We'll use a standard ChatML-style format which works well with most LLMs
    return (
        f"Context: The user is feeling {emotion}. "
        f"Task: Respond as an empathetic assistant.\n"
        f"User: {row['user_input']}\n"
        f"Assistant:"
    )

print("Creating prompts with randomized emotion labels...")
df["prompt"] = df.apply(build_prompt, axis=1)
df["target"] = df["bot_response"]

# Token estimation & filter
df["prompt_tokens"] = df["prompt"].apply(estimate_tokens)
df["target_tokens"] = df["target"].apply(estimate_tokens)
df = df[(df["prompt_tokens"] < MAX_PROMPT_TOKENS) & (df["target_tokens"] < MAX_TARGET_TOKENS)]
print(f"After token filtering: {len(df)} rows")

# Split (80/10/10)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["user_emotion"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["user_emotion"])

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Save
train_df[["prompt", "target", "language", "user_emotion"]].to_json("train_en.jsonl", orient="records", lines=True, force_ascii=False)
val_df[["prompt", "target", "language", "user_emotion"]].to_json("val_en.jsonl", orient="records", lines=True, force_ascii=False)
test_df[["prompt", "target", "language", "user_emotion"]].to_json("test_en.jsonl", orient="records", lines=True, force_ascii=False)

print("\nSaved: train_en.jsonl, val_en.jsonl, test_en.jsonl")
!ls -lh *.jsonl

Loading en dataset...
/kaggle/input/llm-training-in-multilingual-languages/english_dataset.jsonl → loaded 1000 rows, skipped 0 bad lines

Cleaning text...
After filtering: 999 rows
Creating prompts with randomized emotion labels...
After token filtering: 999 rows
Train: 799, Val: 100, Test: 100

Saved: train_en.jsonl, val_en.jsonl, test_en.jsonl
-rw-r--r-- 1 root root  31K Feb  4 14:23 test_en.jsonl
-rw-r--r-- 1 root root 246K Feb  4 14:23 train_en.jsonl
-rw-r--r-- 1 root root  31K Feb  4 14:23 val_en.jsonl
